> 📄 English: [basic_usage.ipynb](basic_usage.ipynb)

# Ollama 基础用法示例

这里展示了用 Ollama 替代 OpenAI / 其他 API 调用的最简单方式。
非常适合在进入 Agent 之前理解核心概念。

1) 直接 API 调用
2) Ollama API 参数
3) 基础 LangChain 集成
4) 对话


In [ ]:
import requests
import time

# 让我们检查一下 Ollama 是否正在运行：


In [ ]:
def check_ollama_running():
    """Check if Ollama is available."""
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=5)
        return response.status_code == 200
    except:
        return False

if not check_ollama_running():
    print("❌ Ollama not running!")
    print("Start it with: ollama serve")
    print("And make sure you have a model: ollama pull llama3.1:8b")
else:
    print("✅ Ollama is running\n")
    print("🚀 Basic Ollama Usage Examples")
    print("=" * 40)

# 直接调用 API


# system vs user 调用：
✅ role: "system"
用途：为模型设定初始行为、语气或约束。
示例用法：在对话开始前定义助手的个性、职责范围或指令。
模型会读取一次，并用它来影响后续所有回复。
在大多数应用中不会展示给用户。

✅ role: "user"
用途：代表用户提出的实际问题或输入。
触发模型做出回复。
之后可以跟随 assistant 消息来延续来回对话。


In [ ]:
def example_1_direct_api(sys_promt, usr_promt):
    # This replaces OpenAI API calls
    # response = requests.post("http://localhost:11434/api/chat", json={
    #     "model": "llama3.1:8b",
    #     "messages": [
    #         {"role": "system", "content": "You are a helpful assistant."},
    #         {"role": "user", "content": "What is Python?"}
    #     ]
    # })
    response = requests.post("http://localhost:11434/api/chat", json={
        "model": "llama3.1:8b",
        "messages": [{"role": "system", "content": sys_promt},
                     {"role": "user", "content":  usr_promt}
                     ],
        "stream": False  # Ensure it's not streaming
    })
    if response.status_code == 200:
        result = response.json()
        print(f"Response: {result["message"]["content"]}")
    else:
        print(f"Error: {response.status_code}")

In [ ]:
print("Example 1: Direct API Call")
print("-" * 30)
sys_promt = "You are a helpful assistant."
usr_promt = "What is Python?"
example_1_direct_api(sys_promt,usr_promt)

## Ollama API 参数

Ollama 提供了丰富的自定义选项。以下是常用参数：

| 参数 | 类型 | 默认值 | 描述 |
|-----------|------|---------|-------------|
| `model` | string | - | **必填。** 模型标识符（例如 `"llama3.1:8b"`） |
| `messages` | array | - | `/api/chat` 的聊天对话 |
| `stream` | boolean | true | 生成时流式返回响应 |
| `temperature` | float | 0.8 | 随机性（0.0 = 确定性，2.0 = 非常随机） |
| `top_p` | float | 0.9 | 核采样（0.1 = 仅 top 10% tokens） |
| `num_predict` | int | 128 | 生成的最大 token 数（-1 = 不限） |
| `repeat_penalty` | float | 1.1 | 对重复 token 的惩罚 |
| `system` | string | - | 用于引导行为的 system 消息 |
| `stop` | array | - | 在这些字符串处停止生成 |


In [ ]:
def example_2_with_parameters(temperature, num_predict, usr_promt):
   
    response = requests.post("http://localhost:11434/api/chat", json={
        "model": "llama3.1:8b",
        "messages": [{"role": "user", "content": usr_promt}],
        "temperature": temperature,  # Higher temperature for creativity
        "num_predict": num_predict,   # Limit response length
        "stream": False  # Ensure it's not streaming
    })
    
    if response.status_code == 200:
        result = response.json()["message"]["content"]
        print(f"Creative response: {result}")
    else:
        print(f"Error: {response.status_code}")

In [ ]:
print("\nExample 2: With Parameters")
print("-" * 30)
usr_promt = "Write a creative story about a robot."
temperature = 0.8
num_predict = 100
example_2_with_parameters(temperature, num_predict, usr_promt)

# LangChain 集成

| 特性 | ChatOllama | 直接 API |
|---------|------------|------------|
| `抽象层级` | 高 | 低 |
| `易用性` | 更简单 | 需要手动格式化 |
| `自定义（headers 等）` | 有限 | 完整 |
| `依赖` | 需要 LangChain | 仅需 requests |
| `最适合` | RAG 流水线、快速原型 | 自定义工具、底层控制 |


In [ ]:
def example_3_langchain_basic(usr_promt):    
    try:
        from langchain_community.chat_models import ChatOllama
        
        # Replace ChatOpenAI with ChatOllama
        llm = ChatOllama(model="llama3.1:8b", temperature=0.1)
        
        response = llm.invoke(usr_promt)
        print(f"LangChain response: {response.content}")
        
    except ImportError:
        print("LangChain not installed. Run: pip install langchain langchain-community")
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
print("\nExample 3: LangChain Integration")
print("-" * 30)
usr_promt = "Explain machine learning in one sentence."
example_3_langchain_basic(usr_promt)

# 维护对话上下文


在与语言模型的对话中，消息会在不同角色之间交换。"assistant" 角色代表模型对用户输入的响应。当我们把 {"role": "assistant", "content": assistant_msg} 加入消息历史时，就是在存储模型上一次的回复。这有助于在多轮对话中维护上下文，让模型记住自己之前说过的话并据此做出回应。


In [ ]:
def example_4_conversation(messages):   
    response = requests.post("http://localhost:11434/api/chat", json={
        "model": "llama3.1:8b",
        "messages": messages,
        "stream": False
    })
    
    if response.status_code == 200:
        assistant_msg = response.json()["message"]["content"]
        return assistant_msg

### 开始对话：


In [ ]:
print("\nExample 4: Conversation")
print("-" * 30)
messages = [{"role": "system", "content": "You are a helpful coding assistant."}]
messages.append({"role": "user", "content": "How do I read a file in Python?"})
assistant_msg = example_4_conversation(messages)
print(f"Assistant:\n{assistant_msg}")

### 添加 assistant：


In [ ]:
messages.append({"role": "assistant", "content": assistant_msg})

### 添加追问：


In [ ]:
messages.append({"role": "user", "content": "What about error handling?"})
answer = example_4_conversation(messages)
print(f"\n\nFollow-up:\n{answer}")

✅ 所有示例已完成！

💡 要点：

• 将 OpenAI 的 URL 替换为 http://localhost:11434

• 不需要 API key！

• 在 LangChain 中用 ChatOllama 替换 ChatOpenAI
